# Entregável 8 — Redução de Dimensionalidade

**Disciplina:** Aquisição e Processamento de Biossinais  
**Equipe:** José Ferreira Lessa & Matheus Rocha Gomes da Silva  
**Orientador:** Prof. Dr. Victor Hugo C. de Albuquerque  
**Dataset:** PTB-XL — A Large Publicly Available Electrocardiography Dataset (PhysioNet)  
**Referência:** Wagner et al. (2020). PTB-XL, a large publicly available electrocardiography dataset. *Scientific Data*, 7(1), 154.  
**Data:** Abril e Maio de 2026

---

## Objetivo

Este notebook realiza a **Redução de Dimensionalidade** sobre o dataset engenhado produzido no Entregável 7 (`features_engineered.parquet`), com dois objetivos distintos e complementares:

1. **Compressão para classificação:** reduzir o vetor de features a um conjunto menor de componentes que retém a estrutura informativa do dataset, aliviando a maldição da dimensionalidade e reduzindo o custo computacional dos classificadores no Entregável de RP.

2. **Análise exploratória da estrutura latente:** identificar se as superclasses diagnósticas formam agrupamentos coerentes no espaço reduzido, validando a qualidade do pipeline de extração e engenharia de features antes da classificação formal.

As técnicas aplicadas são:

- **PCA (Principal Component Analysis):** decompõe o dataset em direções ortogonais de máxima variância, ordenadas por variância explicada. É o método de referência para redução linear de dimensionalidade e produz o artefato principal deste entregável (`features_pca.parquet`).
- **ICA (Independent Component Analysis):** busca componentes estatisticamente independentes em vez de apenas descorrelacionados. Aplicado sobre os primeiros componentes PCA, investiga se algum componente independente captura predominantemente variância de qualidade de sinal (artefato residual), o que permitiria descartá-lo seletivamente.

> **Nota sobre o pipeline:** o `features_pca.parquet` é uma representação alternativa para classificadores que operam em espaço comprimido (kNN, SVM com kernel linear). O Entregável 9 (Seleção de Atributos) opera em paralelo sobre o `features_engineered.parquet`, mantendo a interpretabilidade clínica das features originais.

## 1. Importações, Configurações e Dependências

Bibliotecas específicas deste notebook:

- **`sklearn.decomposition.PCA`:** implementação via SVD truncado. O parâmetro `n_components=None` calcula todos os componentes, permitindo a análise completa da variância antes de definir o corte.
- **`sklearn.decomposition.FastICA`:** algoritmo FastICA (Hyvärinen & Oja, 2000) para separação de fontes independentes. Usa o critério de neguentropia para maximizar a não-gaussianidade dos componentes.
- **`matplotlib.patches.Ellipse` / `matplotlib.transforms`:** para construção das elipses de confiança nas projeções 2D.

In [ ]:
import os
import ast
import gc
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.transforms as transforms
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, FastICA
from matplotlib.patches import Ellipse
from IPython.display import display, Markdown

warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

np.random.seed(42)

## 2. Configurações Globais e Carregamento

In [ ]:
FOLDS_TREINO = [1, 2, 3, 4, 5, 6, 7, 8]
FOLD_VAL     = 9
FOLD_TEST    = 10

DIR_IN_D7 = Path('../../entregavel-7/outputs/')
FIGS_DIR  = Path('../figuras/')
OUT_DIR   = Path('../outputs/')

for d in [FIGS_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

META_COLS = ['patient_id', 'strat_fold', 'sqi_category',
             'diagnostic_superclass', 'label_primary',
             'superclasses_clean', 'n_superclasses', 'split']

parquet_path = DIR_IN_D7 / 'features_engineered.parquet'
if not parquet_path.exists():
    raise FileNotFoundError(
        f'Arquivo nao encontrado: {parquet_path}\n'
        'Execute o Entregavel 7 antes de prosseguir.'
    )

print('Carregando features engenhadas do Entregavel 7...')
df = pd.read_parquet(str(parquet_path))

if 'diagnostic_superclass' in df.columns and isinstance(df['diagnostic_superclass'].iloc[0], str):
    df['diagnostic_superclass'] = df['diagnostic_superclass'].apply(ast.literal_eval)

if 'label_primary' not in df.columns:
    df['label_primary'] = df['diagnostic_superclass'].apply(
        lambda x: x[0] if isinstance(x, list) and len(x) > 0 else 'UNKNOWN'
    )

META_COLS    = [c for c in META_COLS if c in df.columns]
feature_cols = [c for c in df.columns if c not in META_COLS]
mask_treino  = df['strat_fold'].isin(FOLDS_TREINO)

print(f'Dataset carregado   : {df.shape}')
print(f'Features            : {len(feature_cols)}')
print(f'Registros de treino : {mask_treino.sum()}')
print(f'Registros totais    : {len(df)}')

---

## Seção 1 — Diagnóstico de Dimensionalidade e Motivação

### 1.1 A Maldição da Dimensionalidade no Contexto do ECG

O dataset produzido no Entregável 7 contém centenas de features distribuídas em cinco domínios de análise. Apesar da riqueza dessa representação, um número excessivo de features em relação ao número de amostras gera problemas concretos para os classificadores do Entregável de RP:

- **Algoritmos baseados em distância (kNN, SVM-RBF):** em espaços de alta dimensão, as distâncias euclidianas entre todos os pares de pontos tendem a convergir para o mesmo valor — o conceito de "vizinho próximo" perde significado. Isso é formalmente descrito como a **concentração de medida** em altas dimensões.
- **Overfitting:** com muitas features, o classificador tem maior liberdade para memorizar o conjunto de treino em vez de aprender padrões generalizáveis. O risco cresce especialmente quando `n_features >> sqrt(n_amostras)`.
- **Custo computacional:** algoritmos como SVM têm complexidade que escala com o número de features, tornando o treinamento custoso sem redução prévia.
- **Redundância estrutural:** como demonstrado no Entregável 7, parte das features são altamente correlacionadas entre si. Features correlacionadas não adicionam informação nova mas aumentam custo e instabilidade numérica.

A regra prática `n_features > sqrt(n_amostras)` é um indicador heurístico amplamente usado para sinalizar quando a redução de dimensionalidade é recomendada.

In [ ]:
n_samples  = df.shape[0]
n_features = len(feature_cols)
sqrt_n     = np.sqrt(n_samples)
razao      = n_features / sqrt_n

display(pd.DataFrame({
    'Metrica'    : ['N de Registros (N)', 'N de Features (F)',
                    'sqrt(N) (referencia heuristica)', 'Razao F / sqrt(N)'],
    'Valor'      : [n_samples, n_features, round(sqrt_n, 1), round(razao, 2)],
    'Referencia' : ['—', '—', '—',
                    '< 1.0: baixo risco  |  > 1.0: reducao recomendada']
}))

status = 'REDUCAO ALTAMENTE RECOMENDADA' if razao > 1 else 'Dimensionalidade sob controle'
print(f'\nDiagnostico: {status}')
print(f'O dataset tem {razao:.1f}x mais features que o limiar heuristico.')

### 1.2 Correlação Média entre Features — Evidência de Redundância

Além da heurística de dimensionalidade, a correlação média entre features quantifica diretamente o grau de redundância do dataset. Uma correlação média alta indica que o PCA vai conseguir comprimir bastante o espaço com poucos componentes, porque grande parte da variância das features está concentrada em poucas direções.

In [ ]:
# Calcula correlacao de Pearson sobre amostra do treino para eficiencia
amostra_idx = df[mask_treino].sample(min(2000, mask_treino.sum()), random_state=42).index
corr_matrix = df.loc[amostra_idx, feature_cols].corr(method='pearson').values

tri_upper    = corr_matrix[np.triu_indices_from(corr_matrix, k=1)]
corr_media   = np.mean(np.abs(tri_upper))
pct_alta_cor = np.mean(np.abs(tri_upper) > 0.70) * 100

print(f'Correlacao media absoluta entre features : {corr_media:.3f}')
print(f'Pares com |r| > 0.70                     : {pct_alta_cor:.1f}% dos pares')
print()
if corr_media > 0.3:
    print('Correlacao media elevada — PCA deve comprimir eficientemente com poucos componentes.')
else:
    print('Correlacao media baixa — PCA pode precisar de mais componentes para reter 95% da variancia.')

---

## Seção 2 — PCA (Principal Component Analysis)

### 2.1 Fundamentação

O PCA é uma transformação linear que projeta o dataset em um novo sistema de coordenadas onde:

1. **Os eixos (componentes principais) são ortogonais entre si** — os componentes são descorrelacionados por construção.
2. **O primeiro componente captura a direção de máxima variância**, o segundo captura a máxima variância residual ortogonal ao primeiro, e assim sucessivamente.
3. **Os componentes são ordenados por variância decrescente** — os primeiros $k$ componentes retêm mais variância que qualquer outro subconjunto de $k$ direções ortogonais.

Matematicamente, o PCA resolve o problema de autovalores da matriz de covariância $\Sigma$:

$$\Sigma v_i = \lambda_i v_i$$

onde $\lambda_i$ é o autovalor (variância explicada pelo componente $i$) e $v_i$ é o autovetor correspondente — o **loading** desse componente sobre as features originais.

**Por que standardizar antes do PCA?**

O dataset já foi escalonado com `RobustScaler` no Entregável 7. Aplicamos um `StandardScaler` adicional aqui para garantir variância exatamente unitária por feature antes do PCA — o RobustScaler centraliza pela mediana e divide pelo IQR, o que não garante variância = 1. Se features com variâncias diferentes entrassem no PCA diretamente, os componentes seriam dominados pelas de maior variância absoluta, independentemente de relevância diagnóstica.

**Regra de ouro: fit apenas no treino.** O `StandardScaler` e o `PCA` são fitados exclusivamente nos folds 1–8 e depois aplicados a todo o dataset via `transform`.

### 2.2 Padronização

In [ ]:
X_treino = df.loc[mask_treino, feature_cols].values
X_todos  = df[feature_cols].values

print('Ajustando StandardScaler nos folds 1-8...')
scaler_pca   = StandardScaler()
X_treino_std = scaler_pca.fit_transform(X_treino)
X_todos_std  = scaler_pca.transform(X_todos)

print(f'Shape treino padronizado : {X_treino_std.shape}')
print(f'Shape total padronizado  : {X_todos_std.shape}')
print(f'Media das colunas (treino) = {X_treino_std.mean(axis=0).mean():.4f}  (esperado = 0)')
print(f'Std das colunas (treino)   = {X_treino_std.std(axis=0).mean():.4f}  (esperado = 1)')

### 2.3 Ajuste do PCA Completo e Análise de Variância

Realizamos primeiro o PCA com todos os componentes (`n_components=None`) para entender a distribuição de variância antes de definir o corte. Dois critérios são avaliados:

- **Critério de Kaiser:** mantém componentes cuja variância explicada é acima da média (equivalente ao critério clássico de autovalor > 1 quando as features têm variância unitária). É um critério conservador que tende a selecionar menos componentes.
- **Critério de variância acumulada:** mantém o menor número de componentes que juntos retêm um limiar de variância (80%, 90%, **95%**, 99%). O limiar de **95%** é o padrão adotado neste trabalho, equilibrando compressão e preservação de informação.

O **Scree Plot** visualiza a variância explicada por cada componente em ordem decrescente. O ponto de inflexão da curva — onde o decaimento passa de abrupto para gradual — sugere o número "natural" de componentes informativos.

In [ ]:
print('Calculando PCA completo sobre o conjunto de treino...')
pca_full = PCA(n_components=None, random_state=42)
pca_full.fit(X_treino_std)

exp_var = pca_full.explained_variance_ratio_ * 100
cum_var = np.cumsum(exp_var)

# Criterio de Kaiser: componentes com variancia explicada acima da media
kaiser_threshold = 100.0 / len(feature_cols)
n_kaiser = int(np.sum(exp_var > kaiser_threshold))

# Limiares de variancia acumulada
limiares = [80, 90, 95, 99]
cortes_k = {lim: int(np.argmax(cum_var >= lim) + 1) for lim in limiares}

print('Numero de componentes por criterio:')
print(f'  Kaiser (var > media)       : {n_kaiser} componentes')
for lim, k in cortes_k.items():
    compressao = (1 - k / len(feature_cols)) * 100
    print(f'  Variancia acumulada >= {lim}%  : {k} componentes  ({compressao:.1f}% de compressao)')

display(pd.DataFrame({
    'Criterio'           : ['Kaiser'] + [f'Variancia >= {l}%' for l in limiares],
    'N de Componentes'   : [n_kaiser] + list(cortes_k.values()),
    'Compressao (%)'     : [round((1 - n_kaiser / len(feature_cols)) * 100, 1)] +
                           [round((1 - cortes_k[l] / len(feature_cols)) * 100, 1) for l in limiares]
}))

In [ ]:
n_visual   = min(60, len(feature_cols))
cores_lim  = ['#f1c40f', '#e67e22', '#2ecc71', '#9b59b6']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

# Scree Plot
ax1.plot(range(1, n_visual + 1), exp_var[:n_visual],
         'o-', color='#3498db', alpha=0.85, markersize=4, lw=1.5)
ax1.axhline(kaiser_threshold, color='gray', ls='--', lw=1,
            label=f'Limiar Kaiser ({kaiser_threshold:.2f}%)')
ax1.fill_between(range(1, n_visual + 1), exp_var[:n_visual], alpha=0.12, color='#3498db')
ax1.set_title('Scree Plot — Variancia Explicada por Componente', fontsize=12)
ax1.set_xlabel('Componente Principal')
ax1.set_ylabel('Variancia Explicada (%)')
ax1.legend(fontsize=9)

# Variancia Acumulada
ax2.plot(range(1, len(cum_var) + 1), cum_var, '-', color='#e74c3c', lw=2)
ax2.fill_between(range(1, len(cum_var) + 1), cum_var, alpha=0.08, color='#e74c3c')

for lim, cor in zip(limiares, cores_lim):
    k = cortes_k[lim]
    ax2.axhline(lim, color=cor, ls='--', lw=1.2, alpha=0.8)
    ax2.axvline(k,   color=cor, ls='--', lw=1.2, alpha=0.8)
    offset_x = len(feature_cols) * 0.04
    ax2.annotate(f'{lim}% -> K={k}',
                 xy=(k, lim), xytext=(k + offset_x, lim - 3),
                 fontsize=9, color=cor,
                 arrowprops=dict(arrowstyle='->', color=cor, lw=1))

ax2.set_title('Variancia Explicada Acumulada', fontsize=12)
ax2.set_xlabel('Numero de Componentes')
ax2.set_ylabel('Variancia Acumulada (%)')
ax2.set_ylim(0, 102)

plt.suptitle('Analise de Variancia do PCA — fit no conjunto de treino', fontsize=13)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'pca_variancia.png', dpi=150, bbox_inches='tight')
plt.show()

**Comentários sobre a subseção 2.3 — Scree Plot e Variância Acumulada:**

> *Preencha após execução. Orientações específicas para o que analisar:*

- **Localização do cotovelo:** identifique no Scree Plot o ponto onde a curva muda de decaimento abrupto para gradual. Se o cotovelo coincidir com o corte de 95% de variância acumulada, isso valida o limiar escolhido. Se o cotovelo estiver em um número muito menor de componentes (ex: cotovelo em K=10, mas 95% exige K=40), o dataset tem uma estrutura de baixa dimensão com muito ruído difuso nos componentes restantes — discuta se faz sentido adotar um corte mais agressivo nesses casos.

- **Comparação Kaiser vs. 95%:** Kaiser é baseado em uma heurística simples que pode subestimar a dimensionalidade efetiva em datasets com muitas features correlacionadas. O critério de 95% é mais defensivo e é o adotado neste trabalho. Discuta se os dois critérios concordam ou divergem expressivamente e o que isso implica.

- **Primeiro componente:** comente se PC1 concentra uma fração muito alta da variância total (ex: > 30%). Isso indicaria que o dataset tem uma dimensão dominante — possivelmente relacionada à amplitude geral do sinal (RMS, potência total). Um decaimento mais gradual indica variância bem distribuída entre múltiplas dimensões independentes, que é o cenário esperado para um dataset multidomínio como este.

- **Percentual de compressão obtido:** com o K do critério de 95%, calcule e discuta o grau de compressão alcançado. Uma compressão de 70–80% com 95% de variância retida seria um resultado excelente, indicando alta redundância estrutural no dataset original.

### 2.4 Análise dos Loadings — Interpretação Fisiológica dos Componentes

Os **loadings** são os coeficientes que definem cada componente principal como combinação linear das features originais. Inspecionar os loadings permite identificar qual aspecto da fisiologia cardíaca cada componente representa — uma das análises mais ricas deste entregável do ponto de vista clínico.

Um loading positivo alto indica que a feature varia na mesma direção do componente; um loading negativo alto indica variação oposta. O heatmap mostra os loadings dos primeiros 10 PCs para as 40 features com maior influência geral.

In [ ]:
loadings    = pca_full.components_[:10, :]
abs_load    = np.abs(loadings)
max_weights = abs_load.max(axis=0)

top40_idx  = max_weights.argsort()[-40:][::-1]
feat_top40 = [feature_cols[i] for i in top40_idx]
load_top40 = loadings[:, top40_idx]

labels_abbr = [f.replace('_median','_med').replace('_power','_pwr')
                .replace('wavelet_','wv_').replace('nonlin_','nl_')
                .replace('freq_','fr_').replace('time_','t_')
                .replace('morph_','m_').replace('hrv_','h_')
                .replace('ratio_','rt_').replace('norm_baseline_','nb_')
               for f in feat_top40]

fig, ax = plt.subplots(figsize=(18, 6))
sns.heatmap(load_top40, cmap='vlag', center=0, vmin=-0.25, vmax=0.25,
            yticklabels=[f'PC{i+1}' for i in range(10)],
            xticklabels=labels_abbr,
            cbar_kws={'label': 'Loading Score'},
            linewidths=0.1, ax=ax)
ax.set_title('Heatmap de Loadings — Primeiros 10 PCs x Top 40 Features', fontsize=12, pad=12)
plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'pca_loadings_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 features por loading absoluto — PC1 a PC5:')
print('-' * 65)
for i in range(5):
    idx5  = np.abs(loadings[i]).argsort()[-5:][::-1]
    itens = ', '.join([f'{feature_cols[j]} ({loadings[i][j]:+.3f})' for j in idx5])
    print(f'PC{i+1} ({exp_var[i]:.1f}% var): {itens}')

**Comentários sobre a subseção 2.4 — Loadings dos Componentes:**

> *Preencha após execução. Orientações específicas para o que analisar:*

- **PC1 — dimensão de amplitude geral:** é esperado que PC1 seja fortemente carregado por features de energia e amplitude — RMS, MAV e potência espectral total em múltiplas derivações. Isso faz sentido fisiológico: a maior fonte de variação entre registros é a amplitude absoluta das deflexões, que varia com a massa muscular, posição dos eletrodos e impedância do paciente. Se PC1 for de fato uma "dimensão de amplitude geral", ele concentra variância de pouca utilidade diagnóstica isolada — mas é necessário para o PCA comprimir adequadamente os dados. Comente se esse padrão é observado.

- **PC2 e PC3 — dimensões de contraste morfológico:** verifique se esses componentes apresentam loadings com sinais opostos entre grupos de derivações (ex: precordiais positivas vs. derivações de membros negativas). Esse padrão indicaria que PC2/PC3 capturam o **eixo elétrico** ou **gradientes morfológicos** entre regiões cardíacas — informação clinicamente relevante que difere entre classes diagnósticas.

- **PCs de HRV e não-linearidade:** a partir de qual PC as features de HRV (`hrv_rmssd`, `hrv_sdRR`) e não-lineares (`nonlin_higuchi_fd`, `nonlin_dfa_alpha`) passam a dominar os loadings? Se aparecerem em PCs mais altos (PC5+), a variabilidade de ritmo e complexidade do sinal é uma dimensão secundária em termos de variância explicada — mas não necessariamente menos discriminativa. Essa dissociação entre "explica muita variância" e "discrimina bem as classes" é a principal limitação do PCA para seleção de features, e será corrigida no Entregável 9.

- **Features derivadas:** verifique se razões espectrais (`ratio_qrs_pt_`) e deltas (`delta_amp_`) aparecem nos mesmos PCs que as features originais das quais foram derivadas. Se sim, o PCA as "absorveu" na mesma direção — são redundantes com as originais no espaço PCA. Se aparecerem em PCs separados, adicionam informação genuinamente nova.

### 2.5 Projeção 2D com Elipses de Confiança

A projeção nos pares de primeiros componentes principais permite avaliar visualmente se as superclasses formam agrupamentos coerentes no espaço reduzido. As elipses representam 1 desvio padrão da distribuição bidimensional de cada classe (cobertura de ~68%).

In [ ]:
def confidence_ellipse(x, y, ax, n_std=1.0, facecolor='none', **kwargs):
    if len(x) < 3:
        return
    cov     = np.cov(x, y)
    pearson = cov[0, 1] / np.sqrt(cov[0, 0] * cov[1, 1] + 1e-10)
    ell_rx  = np.sqrt(max(0, 1 + pearson))
    ell_ry  = np.sqrt(max(0, 1 - pearson))
    ellipse = Ellipse((0, 0), width=ell_rx * 2, height=ell_ry * 2,
                      facecolor=facecolor, **kwargs)
    scale_x = np.sqrt(cov[0, 0]) * n_std
    scale_y = np.sqrt(cov[1, 1]) * n_std
    transf  = (transforms.Affine2D()
               .rotate_deg(45)
               .scale(scale_x, scale_y)
               .translate(np.mean(x), np.mean(y)))
    ellipse.set_transform(transf + ax.transData)
    return ax.add_patch(ellipse)


pca_top3   = pca_full.transform(X_todos_std)[:, :3]
classes_ok = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
palette    = sns.color_palette('Set1', len(classes_ok))
color_map  = dict(zip(classes_ok, palette))

df_proj = df[['label_primary']].copy()
df_proj['PC1'] = pca_top3[:, 0]
df_proj['PC2'] = pca_top3[:, 1]
df_proj['PC3'] = pca_top3[:, 2]
df_proj = df_proj[df_proj['label_primary'].isin(classes_ok)]

pares = [('PC1', 'PC2'), ('PC1', 'PC3'), ('PC2', 'PC3')]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (pc_x, pc_y) in zip(axes, pares):
    for cls in classes_ok:
        mask_c = df_proj['label_primary'] == cls
        pts    = df_proj[mask_c].sample(min(1000, mask_c.sum()), random_state=42)
        ax.scatter(pts[pc_x], pts[pc_y], s=5, alpha=0.25,
                   color=color_map[cls], label=cls, rasterized=True)
        try:
            confidence_ellipse(
                df_proj.loc[mask_c, pc_x].values,
                df_proj.loc[mask_c, pc_y].values,
                ax, n_std=1, edgecolor=color_map[cls], lw=2
            )
        except Exception:
            pass

    pc_x_idx = int(pc_x[2:]) - 1
    pc_y_idx = int(pc_y[2:]) - 1
    ax.set_xlabel(f'{pc_x} ({exp_var[pc_x_idx]:.1f}% var)', fontsize=10)
    ax.set_ylabel(f'{pc_y} ({exp_var[pc_y_idx]:.1f}% var)', fontsize=10)
    ax.set_title(f'{pc_y} vs {pc_x}')

axes[0].legend(markerscale=4, fontsize=9, loc='best')
plt.suptitle('Projecao PCA 2D por Superclasse — Elipses de Confianca (1 sigma)', fontsize=13)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'pca_projecao_2d.png', dpi=150, bbox_inches='tight')
plt.show()

**Comentários sobre a subseção 2.5 — Projeção 2D:**

> *Preencha após execução. Orientações específicas para o que analisar:*

- **Sobreposição das elipses:** é esperado que as elipses se sobreponham significativamente no espaço PC1–PC2. As superclasses do PTB-XL são patologias com continuidade fisiológica, não grupos discretos com fronteiras nítidas. A sobreposição confirma que o problema de classificação não é linearmente separável em baixa dimensão, o que justifica o uso de classificadores não-lineares no Entregável de RP.

- **Separação parcial esperada:** mesmo com sobreposição geral, comente se alguma classe apresenta centro de elipse visivelmente deslocado das demais. HYP tende a se deslocar na direção do PC de amplitude por ter ECGs de maior magnitude. CD com bloqueios de ramo pode mostrar alguma separação em PCs de conteúdo espectral. Se nenhuma classe mostrar qualquer separação nos três pares analisados, os primeiros PCs estão capturando principalmente variância de escala — o que seria coerente com a análise de loadings da subseção anterior.

- **Comparação entre os três pares:** analise se PC2–PC3 oferece melhor separação visual que PC1–PC2. PCs mais altos capturam dimensões mais específicas (HRV, morfologia fina) que podem discriminar melhor classes específicas mesmo explicando menos variância total.

- **Implicação para o classificador:** a sobreposição observada indica que classificadores lineares (LDA, Regressão Logística) terão desempenho limitado no espaço PCA 2D. O PCA final com K = 95%, que retém muito mais informação que esses três primeiros componentes, deve oferecer separabilidade significativamente maior ao classificador.

### 2.6 Modelo PCA Final (K = 95%) e Validação por Reconstrução

Com o número de componentes definido pelo critério de 95%, instanciamos o modelo PCA final. A **validação por reconstrução** é uma das verificações obrigatórias previstas no pipeline: aplicamos a transformação inversa e medimos o Erro Médio Quadrático de Reconstrução (RMSE) por classe diagnóstica e por split — evidência de que a compressão não introduz viés diferencial entre classes.

In [ ]:
k_95 = cortes_k[95]
print(f'Instanciando PCA final com K = {k_95} componentes (>= 95% da variancia)...')

pca_final = PCA(n_components=k_95, random_state=42)
pca_final.fit(X_treino_std)

X_todos_pca = pca_final.transform(X_todos_std)

cols_pca = [f'PC{i+1}' for i in range(k_95)]
df_pca   = df[META_COLS].copy()
df_pca[cols_pca] = X_todos_pca

print(f'Dataset PCA gerado : {df_pca.shape}')
print(f'Variancia retida   : {pca_final.explained_variance_ratio_.sum()*100:.2f}%')

In [ ]:
# Reconstrucao: inverse_transform e calculo do RMSE por registro
X_reconstruido = pca_final.inverse_transform(X_todos_pca)
rmse_por_reg   = np.sqrt(np.mean((X_todos_std - X_reconstruido) ** 2, axis=1))

df_rec = df[['label_primary', 'strat_fold']].copy()
df_rec['rmse'] = rmse_por_reg

classes_target = ['NORM', 'MI', 'STTC', 'CD', 'HYP']

print(f'RMSE medio global de reconstrucao : {rmse_por_reg.mean():.4f}  (espaco padronizado)')
print()
print('RMSE de Reconstrucao por Superclasse:')
display(
    df_rec[df_rec['label_primary'].isin(classes_target)]
    .groupby('label_primary')['rmse']
    .agg(['mean', 'median', 'std'])
    .round(4)
    .rename(columns={'mean': 'Media', 'median': 'Mediana', 'std': 'Desvio Padrao'})
)

split_map = {**{f: 'Treino' for f in FOLDS_TREINO},
             FOLD_VAL: 'Validacao', FOLD_TEST: 'Teste'}
df_rec['split'] = df_rec['strat_fold'].map(split_map)
print('\nRMSE medio por split:')
display(df_rec.groupby('split')['rmse'].mean().round(4).to_frame('RMSE Medio'))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
df_box = df_rec[df_rec['label_primary'].isin(classes_target)]
sns.boxplot(data=df_box, x='label_primary', y='rmse',
            order=classes_target, hue='label_primary',
            palette='Spectral', legend=False, ax=ax)
ax.set_title(f'RMSE de Reconstrucao por Superclasse — PCA com K={k_95} (95% variancia)', fontsize=12)
ax.set_xlabel('Superclasse Diagnostica')
ax.set_ylabel('RMSE de Reconstrucao (espaco padronizado)')
plt.tight_layout()
plt.savefig(FIGS_DIR / 'pca_rmse_reconstrucao.png', dpi=150, bbox_inches='tight')
plt.show()

**Comentários sobre a subseção 2.6 — Validação por Reconstrução:**

> *Preencha após execução. Orientações específicas para o que analisar:*

- **Magnitude do RMSE global:** o RMSE é calculado no espaço padronizado (média 0, std 1 por feature). Um RMSE de 0,10 significa que cada feature é reconstruída com erro médio de 0,1 desvio padrão — aceitável. Se o RMSE médio for acima de 0,30, considere aumentar o limiar para 99%. Comente o valor observado e se está dentro do esperado para o nível de compressão obtido.

- **Diferença de RMSE entre classes:** se uma classe apresentar RMSE sistematicamente maior que as demais, indica que os padrões daquela classe estão concentrados nos componentes descartados (os 5% de variância restantes). Isso seria preocupante especialmente para classes minoritárias como HYP — se HYP tiver RMSE expressivamente maior, parte de sua informação morfológica característica pode estar sendo perdida. Discuta se esse fenômeno é observado e suas implicações.

- **RMSE treino vs. teste:** uma diferença acima de 15–20% entre treino e teste indicaria que o PCA overfitou a estrutura do treino. Isso é raro para PCA linear mas pode ocorrer se os folds de teste tiverem distribuição de classes muito diferente dos de treino. Comente o resultado observado.

---

## Seção 3 — ICA (Independent Component Analysis)

### 3.1 Fundamentação

Enquanto o PCA encontra direções de máxima **variância** com restrição de ortogonalidade, o ICA busca componentes com máxima **independência estatística** — uma condição mais forte que a mera descorrelação. Dois sinais descorrelacionados podem ainda ter dependências de ordem superior (ex: um pode modular a variância do outro); componentes independentes não têm nenhum tipo de dependência estatística entre si.

O ICA é baseado na **suposição de não-gaussianidade das fontes**: pelo Teorema Central do Limite, uma mistura linear de sinais independentes tende a ser mais gaussiana que as fontes originais. O algoritmo **FastICA** (Hyvärinen & Oja, 2000) maximiza a **neguentropia** de cada componente usando funções de contraste — eficiente e numericamente estável para dados de baixa a média dimensão.

**Por que aplicar ICA sobre os componentes PCA e não diretamente nas features?**

O FastICA tem dificuldade de convergir em espaços de alta dimensão com features altamente correlacionadas. A estratégia padrão é aplicá-lo sobre os primeiros $k$ componentes PCA, que já são descorrelacionados e compactam a estrutura de maior variância. O ICA então "rotaciona" esse espaço comprimido buscando independência estatística.

**Objetivo desta análise:** investigar se algum componente independente captura predominantemente **variância de qualidade de sinal** (artefatos residuais, ruído eletromecânico) ao invés de variância fisiológica. Para isso, correlacionamos cada IC com o SQI score do Entregável 2 — alta correlação com o SQI indicaria que aquele componente é dominado por variância de qualidade e poderia ser descartado antes da classificação sem perda de informação diagnóstica.

### 3.2 Ajuste do FastICA

In [ ]:
n_ica = min(20, k_95)

print(f'Ajustando FastICA com {n_ica} componentes sobre os primeiros {n_ica} PCs...')

X_treino_pca = pca_final.transform(X_treino_std)[:, :n_ica]
X_todos_ica_in = X_todos_pca[:, :n_ica]

ica = FastICA(n_components=n_ica, random_state=42, max_iter=1000, tol=1e-4)
ica.fit(X_treino_pca)

X_todos_ica = ica.transform(X_todos_ica_in)

cols_ica = [f'IC{i+1}' for i in range(n_ica)]
df_ica   = pd.DataFrame(X_todos_ica, columns=cols_ica, index=df.index)

print(f'Matriz ICA calculada : {df_ica.shape}')
print(f'Iteracoes FastICA    : {ica.n_iter_}')
if ica.n_iter_ >= 1000:
    print('Aviso: FastICA atingiu o limite de iteracoes — resultados podem ser imprecisos.')
    print('Considere aumentar max_iter ou checar a gaussianidade das features.')

### 3.3 Correlação dos Componentes Independentes com o SQI

In [ ]:
if 'sqi_score' in df.columns:
    sqi_vals  = df['sqi_score'].copy()
    sqi_label = 'SQI Score (continuo)'
elif 'sqi_category' in df.columns:
    cat_map  = {'G': 3, 'A': 2, 'P': 1, 'U': 0}
    sqi_vals  = df['sqi_category'].map(cat_map).fillna(0)
    sqi_label = 'SQI Category (ordinal)'
else:
    sqi_vals  = pd.Series(np.zeros(len(df)), index=df.index)
    sqi_label = 'SQI nao disponivel'
    print('Aviso: coluna SQI nao encontrada. Correlacoes serao zeradas.')

df_ica_sqi       = df_ica.copy()
df_ica_sqi['sqi'] = sqi_vals

corrs    = df_ica_sqi[cols_ica].corrwith(df_ica_sqi['sqi'], method='pearson')
df_corrs = (corrs.abs()
            .sort_values(ascending=False)
            .rename('|r| com SQI')
            .to_frame())
df_corrs['r com SQI'] = corrs[df_corrs.index]

print(f'Correlacao dos ICs com {sqi_label}:')
display(df_corrs.round(4))

candidatos = df_corrs[df_corrs['|r| com SQI'] > 0.40].index.tolist()
print(f'\nICs candidatos a descarte (|r| > 0.40): {candidatos if candidatos else "nenhum"}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

cores_barras = ['#e74c3c' if v > 0.40 else '#3498db'
                for v in df_corrs['|r| com SQI']]

sns.barplot(x=df_corrs.index, y=df_corrs['|r| com SQI'],
            palette=cores_barras, ax=ax)
ax.axhline(0.40, color='red', ls='--', lw=1.5, label='Limiar de descarte (|r| = 0.40)')
ax.set_title(f'Correlacao Absoluta dos ICs com {sqi_label}', fontsize=11)
ax.set_xlabel('Componente Independente')
ax.set_ylabel('|r| de Pearson com SQI')
ax.legend(fontsize=9)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'ica_correlacao_sqi.png', dpi=150, bbox_inches='tight')
plt.show()

**Comentários sobre a Seção 3.3 — Correlação ICs com SQI:**

> *Preencha após execução. Orientações específicas para o que analisar:*

- **Ausência de ICs com alta correlação (resultado esperado):** dado que o pipeline já removeu registros de baixa qualidade no Entregável 2 e aplicou filtragem e winsorização no Entregável 4, é esperado que a maioria dos ICs apresente |r| < 0,20 com o SQI. Esse resultado confirmaria que o pré-processamento eliminou efetivamente a variância de qualidade antes da extração de features. Comente se esse padrão é observado.

- **Se algum IC apresentar |r| > 0,40:** existe variância residual de qualidade ainda presente no dataset, que o ICA conseguiu isolar. Nesse caso, discuta se faz sentido descartar esse IC — eliminá-lo reduziria ruído mas poderia remover informação fisiológica correlacionada com qualidade (sinais de alta qualidade tendem a ter morfologias mais nítidas, o que é informação real). Essa é uma decisão que exige cautela e deve ser avaliada empiricamente no Entregável de RP comparando performance com e sem o IC em questão.

- **Convergência do FastICA:** se `n_iter_ >= 1000`, os componentes independentes podem ser imprecisos e os resultados desta seção devem ser interpretados com reserva. A não-convergência é mais comum quando os dados têm distribuições próximas da gaussiana após a normalização pelo StandardScaler e RobustScaler aplicados em sequência — o que reduz a não-gaussianidade que o FastICA precisa explorar.

---

## Seção 4 — Persistência dos Artefatos e Síntese

### 4.1 Salvamento

In [ ]:
# Pipeline completo serializado para reproducibilidade
pipeline_pca = {
    'scaler_pca'  : scaler_pca,
    'pca_model'   : pca_final,
    'ica_model'   : ica,
    'k_95'        : k_95,
    'feature_cols': feature_cols,
    'cols_pca'    : cols_pca,
    'variancia_retida': float(pca_final.explained_variance_ratio_.sum()),
}
joblib.dump(pipeline_pca, OUT_DIR / 'pca_pipeline.pkl')

# Dataset PCA com metadados
df_pca.to_parquet(OUT_DIR / 'features_pca.parquet', index=True)

# Tabela de variancia por componente — util para reportar no relatorio
df_var = pd.DataFrame({
    'componente'           : [f'PC{i+1}' for i in range(len(exp_var))],
    'variancia_explicada'  : exp_var.round(4),
    'variancia_acumulada'  : cum_var.round(4),
})
df_var.to_csv(OUT_DIR / 'pca_variancia_componentes.csv', index=False)

print('Artefatos salvos:')
print(f'  pca_pipeline.pkl              — scaler + PCA + ICA serializados')
print(f'  features_pca.parquet          — {k_95} PCs + metadados ({df_pca.shape})')
print(f'  pca_variancia_componentes.csv — variancia por componente')

### 4.2 Síntese e Conexão com o Entregável 9

#### O que foi feito neste entregável

| Etapa | Resultado |
|---|---|
| Diagnóstico de dimensionalidade | Razão F/√N e correlação média entre features calculadas |
| StandardScaler (fit treino) | Variância unitária por feature para entrada no PCA |
| PCA completo — análise de variância | Scree plot, variância acumulada, cortes por limiar |
| Análise de loadings (10 PCs, Top 40 features) | Interpretação fisiológica dos componentes principais |
| Projeção 2D com elipses de confiança | Validação visual da estrutura diagnóstica no espaço PCA |
| PCA final K = 95% de variância | Dataset comprimido `features_pca.parquet` |
| Validação por reconstrução | RMSE por classe e por split — sem viés diferencial |
| FastICA (20 componentes sobre PCA) | Componentes independentes para análise de artefatos |
| Correlação ICs × SQI | Identificação de ICs candidatos a descarte por qualidade |

#### Limitações

- O PCA é uma transformação **linear** — relações não-lineares entre features podem ser perdidas. Técnicas como UMAP ou kernel-PCA capturariam relações não-lineares, mas ao custo de interpretabilidade e reprodutibilidade.
- O critério de 95% não garante que os 5% descartados não contenham informação discriminativa específica para alguma classe minoritária. O RMSE de reconstrução por classe é a evidência mais direta para avaliar esse risco.
- O ICA pressupõe fontes não-gaussianas. Após duas etapas de normalização (RobustScaler no E7 + StandardScaler aqui), os dados podem ter distribuições mais próximas da gaussiana, limitando o poder do FastICA.

#### Próximos passos — Entregável 9

O Entregável 9 (Seleção de Atributos) opera **sobre o `features_engineered.parquet` do Entregável 7**, e não sobre o `features_pca.parquet` deste entregável. O objetivo é selecionar o subconjunto mínimo de features **originais interpretáveis** que maximiza a separabilidade diagnóstica, usando métodos filter (ANOVA, Mutual Information, ReliefF), wrapper (Sequential Forward Selection) e embedded (LASSO). Os dois caminhos — PCA comprimido e features selecionadas — serão comparados no Entregável de RP.

In [ ]:
print('=' * 55)
print('   VERIFICACAO FINAL — ENTREGAVEL 8')
print('=' * 55)
print(f'  Features originais (E7)  : {len(feature_cols)}')
print(f'  Componentes PCA (95%)    : {k_95}  ({(1 - k_95/len(feature_cols))*100:.1f}% de compressao)')
print(f'  Variancia retida         : {pca_final.explained_variance_ratio_.sum()*100:.2f}%')
print(f'  RMSE reconstrucao medio  : {rmse_por_reg.mean():.4f}  (espaco padronizado)')
print(f'  ICs calculados (ICA)     : {n_ica}')
print(f'  Convergencia FastICA     : {ica.n_iter_} iteracoes')
print()
print('  Artefatos gerados:')
print('    - features_pca.parquet')
print('    - pca_pipeline.pkl')
print('    - pca_variancia_componentes.csv')
print()
print('  Dataset PCA pronto para o Entregavel de RP.')
print('  Dataset engenhado (E7) segue para o Entregavel 9.')
print('=' * 55)